> **Runtime: Colab only.** This notebook needs `transformers` + `torch`, which do not run under JupyterLite (no Rust/C extensions in Pyodide). On Colab the install is fast and the model is ~330 MB.
>
> [![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/L3GJ0N/course-notebooks-public/blob/main/06-real-attention-distilgpt2.ipynb)
>
> **[Click here to open in Google Colab](https://colab.research.google.com/github/L3GJ0N/course-notebooks-public/blob/main/06-real-attention-distilgpt2.ipynb)**

# Real Attention with `distilgpt2`

**Lecture 5 — Inside the Transformer**

You wrote a working `attention` function in NumPy across notebooks 04 and 05. Now you'll point it at a real, trained transformer — `distilgpt2` (~82 M parameters, 6 layers, 12 heads, GPT-2 tokenizer) — and confirm that the attention block of one chosen layer reproduces the model's output to float-precision.

**Scope.** We verify the **attention sub-layer** (`ln_1` → `c_attn` → per-head attention → `c_proj`) of layer 3 against HuggingFace's forward pass. We do **not** reproduce the full layer (residuals + LN + FFN) — to do so we'd have to redo every prior layer. Instead we use HF's `output_hidden_states=True` to get the residual-stream input *into* layer 3 for free.

**The Conv1D gotcha** (more on this below). HuggingFace GPT-2 stores attention projections in `Conv1D` layers, not `nn.Linear`. The weight shape is `[in, out]`, opposite of `nn.Linear`. The forward call is `x @ W + b` — *no* transpose. Get this wrong and shapes still match, but values are nonsense.

**Outcome.** By the end of this notebook you will:
- Know that the function you wrote is the actual computation real LLMs run, modulo bookkeeping.
- Have visualised the 12 attention heads of one layer on a real sentence and identified which heads do what.
- Have generated a few next-token candidates with temperature + top-k sampling.


## 1. Setup — load `distilgpt2`

Colab pre-installs `torch`. We add `transformers`. Loading the model takes 5–15 seconds; weights are cached after the first load.

In [ ]:
%pip install -q transformers

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

MODEL_NAME = 'distilgpt2'
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, attn_implementation='eager')
model.eval()

cfg = model.config
d = cfg.hidden_size
n_head = cfg.num_attention_heads
n_layer = cfg.num_hidden_layers
d_head = d // n_head
ln_eps = cfg.layer_norm_epsilon

print(f'Model:    {MODEL_NAME}')
print(f'Layers:   {n_layer}')
print(f'd:        {d}')
print(f'n_head:   {n_head}')
print(f'd_head:   {d_head}  (= d / n_head)')
print(f'ln eps:   {ln_eps}')
print(f'params:   {sum(p.numel() for p in model.parameters()):,}')

## 2. Tokenize the sentence

We use `"The cat sat on the mat"` — the lecture's running cat example, lengthened slightly so the heatmaps in section 8 show some structure. GPT-2's tokenizer encodes leading-space-included tokens (`" cat"`, not `"cat"`).

In [ ]:
SENTENCE = 'The cat sat on the mat'
encoded = tokenizer(SENTENCE, return_tensors='pt')
input_ids = encoded.input_ids        # (1, T)
T = input_ids.shape[-1]
tokens = [tokenizer.decode([tid]) for tid in input_ids[0].tolist()]

print(f'Input: {SENTENCE!r}')
print(f'T = {T} tokens')
for i, (tid, tok) in enumerate(zip(input_ids[0].tolist(), tokens)):
    print(f'  pos {i}: id={tid:>5d}  token={tok!r}')

## 3. One forward pass — capture hidden states + the layer-3 attention output

Two things we need from HF:

1. The **input to layer 3's attention** — that is `hidden_states[3]` (output of layer 2 = input to layer 3). HF returns this for free with `output_hidden_states=True`.
2. The **output of layer 3's attention sub-layer** — what HF computed, so we can compare. Captured via a forward hook on `model.transformer.h[3].attn`.

We also grab `output_attentions=True` so we can compare per-head attention weights in section 8.

Layer index choice: layer 3 (out of 6) — middle of the network, where heads tend to specialise.

In [ ]:
LAYER = 3   # 0-indexed; distilgpt2 has layers 0..5

captured = {}
def hook(module, inputs, outputs):
    # GPT2Attention returns (attn_output, present_key_value, attn_weights_or_None)
    captured['hf_attn_output'] = outputs[0].detach().cpu().numpy()

handle = model.transformer.h[LAYER].attn.register_forward_hook(hook)
with torch.no_grad():
    out = model(**encoded, output_hidden_states=True, output_attentions=True)
handle.remove()

hidden_in = out.hidden_states[LAYER].detach().cpu().numpy()      # (1, T, d) — input to layer LAYER
hf_attn_out = captured['hf_attn_output']                          # (1, T, d) — what we want to reproduce
hf_attn_weights = out.attentions[LAYER].detach().cpu().numpy()    # (1, n_head, T, T)

print(f'hidden_states[{LAYER}] shape: {hidden_in.shape}  — input to layer {LAYER}')
print(f'hf attn output shape:        {hf_attn_out.shape}  — captured by hook')
print(f'hf attn weights shape:       {hf_attn_weights.shape}')

## 4. Extract layer-3 weights as NumPy

Everything we need to reproduce the attention sub-layer:
- `ln_1.weight`, `ln_1.bias` — pre-attention LayerNorm.
- `attn.c_attn.weight`, `attn.c_attn.bias` — packs Q, K, V projections into one `(d, 3d)` matrix.
- `attn.c_proj.weight`, `attn.c_proj.bias` — output projection (the `W_O` from the lecture).

All weights are float32. We keep them as NumPy from here on.

In [ ]:
block = model.transformer.h[LAYER]

ln1_w = block.ln_1.weight.detach().cpu().numpy()
ln1_b = block.ln_1.bias.detach().cpu().numpy()

c_attn_w = block.attn.c_attn.weight.detach().cpu().numpy()
c_attn_b = block.attn.c_attn.bias.detach().cpu().numpy()

c_proj_w = block.attn.c_proj.weight.detach().cpu().numpy()
c_proj_b = block.attn.c_proj.bias.detach().cpu().numpy()

for name, arr in [('ln1.weight', ln1_w), ('ln1.bias', ln1_b),
                  ('c_attn.weight', c_attn_w), ('c_attn.bias', c_attn_b),
                  ('c_proj.weight', c_proj_w), ('c_proj.bias', c_proj_b)]:
    print(f'  {name:18s} shape {str(arr.shape):>14s}  dtype {arr.dtype}')

## 5. The HF Conv1D gotcha

`c_attn` and `c_proj` are HuggingFace `Conv1D` layers (legacy from the original GPT-2 codebase). They are *not* `nn.Linear`. Two things to watch:

- **Weight shape:** `c_attn.weight.shape == (d, 3d)`. With `nn.Linear` it would be `(3d, d)`.
- **Forward:** `output = input @ weight + bias`. With `nn.Linear` it would be `input @ weight.T + bias` (note the transpose).

Get this wrong and your shapes still match, but the numbers are wrong. The cell below is a sanity check — use it as a template the next time you reach into a HuggingFace model.

In [ ]:
# Sanity check: with the right convention the shape works out cleanly
h_for_check = hidden_in[0]                                       # (T, d) — drop batch
qkv_check = h_for_check @ c_attn_w + c_attn_b                    # (T, 3d)
print(f'c_attn applied as x @ W + b → shape {qkv_check.shape}  (expect {(T, 3*d)})')
assert qkv_check.shape == (T, 3 * d)

# What goes wrong with the wrong convention?
try:
    wrong = h_for_check @ c_attn_w.T + c_attn_b
    print(f'  with the wrong .T: shape {wrong.shape}  ← would fail because (d) ≠ (3d) on the inner axis')
except Exception as e:
    print(f'  with the wrong .T: {type(e).__name__}: {e}')

### Your turn: wrap the Conv1D convention as a function

Section 5 showed that HuggingFace's `Conv1D` applies as `x @ weight + bias` — *not* `x @ weight.T + bias` like `nn.Linear`. Wrap that convention as a small helper. We will plug it into the attention block below.

In [ ]:
def apply_conv1d(x, weight, bias):
    """Apply a HuggingFace Conv1D layer: x @ weight + bias.

    Note the `(in, out)` weight layout, opposite of `nn.Linear` which uses `(out, in)`.

    Args:
        x: (..., in) input array
        weight: (in, out) — HF Conv1D weight tensor as NumPy
        bias: (out,) bias vector

    Returns:
        (..., out) output

    Example:
        >>> x = np.array([[1.0, 2.0]])
        >>> W = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])  # shape (2, 3)
        >>> b = np.array([0.1, 0.2, 0.3])
        >>> apply_conv1d(x, W, b)
        array([[1.1, 2.2, 3.3]])
    """
    # TODO: implement using @ and +
    pass

In [ ]:
# --- TEST: apply_conv1d ---

# Docstring example
x_demo = np.array([[1.0, 2.0]])
W_demo = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
b_demo = np.array([0.1, 0.2, 0.3])
out_demo = apply_conv1d(x_demo, W_demo, b_demo)
assert np.allclose(out_demo, [[1.1, 2.2, 3.3]]), f'docstring example failed: got {out_demo}'

# Apply to the real layer-3 c_attn — should match the raw x @ W + b from section 5
qkv_via_helper = apply_conv1d(h_for_check, c_attn_w, c_attn_b)
assert np.allclose(qkv_via_helper, qkv_check, atol=1e-12), 'should match raw x @ W + b'

# Output shape: last axis of weight
assert qkv_via_helper.shape == (T, 3 * d), f'shape should be ({T}, {3*d})'

print('apply_conv1d verified.')

## 6. Build the attention sub-layer in pure NumPy

Five steps, each one a function from notebooks 04 / 05 you have already written.

In [ ]:
def softmax(x, axis=-1):
    x_shifted = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x_shifted)
    return e / e.sum(axis=axis, keepdims=True)

def layernorm(x, gamma, beta, eps):
    """Per-row LayerNorm matching PyTorch (population variance)."""
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

def causal_mask(T):
    M = np.zeros((T, T), dtype=np.float32)
    M[np.triu_indices(T, k=1)] = -np.inf
    return M

In [ ]:
h = hidden_in[0]                          # (T, d) — drop batch
h_ln = layernorm(h, ln1_w, ln1_b, ln_eps)  # (T, d) — ready for the projection

# c_attn produces Q, K, V concatenated along the last dim
qkv = h_ln @ c_attn_w + c_attn_b           # (T, 3d)

# Split contiguously: Q is the first d columns, K is the next d, V is the last d
Q = qkv[:, :d]
K = qkv[:, d:2*d]
V = qkv[:, 2*d:]
print(f'Q, K, V shapes: {Q.shape}, {K.shape}, {V.shape}')

### Your turn: split c_attn output into Q, K, V

Section 6 splits the `(T, 3d)` output of `c_attn` into three `(T, d)` tensors using slicing. Wrap that as a reusable function — it will plug into the attention-block exercise after section 7.

In [ ]:
def split_qkv(qkv, d):
    """Split a (T, 3d) tensor into three (T, d) Q, K, V tensors along the last axis.

    HuggingFace's c_attn packs Q, K, V contiguously: the first d columns are Q,
    the next d are K, the last d are V.

    Args:
        qkv: (T, 3 * d) tensor — the output of c_attn
        d: hidden dimension

    Returns:
        Tuple (Q, K, V), each of shape (T, d).

    Example:
        >>> qkv = np.arange(12).reshape(2, 6)
        >>> Q_, K_, V_ = split_qkv(qkv, d=2)
        >>> Q_.tolist()
        [[0, 1], [6, 7]]
        >>> V_.tolist()
        [[4, 5], [10, 11]]
    """
    # TODO: slice along the last axis to get three (T, d) views
    pass

In [ ]:
# --- TEST: split_qkv ---

# Docstring example
qkv_t = np.arange(12).reshape(2, 6)
Q_t, K_t, V_t = split_qkv(qkv_t, d=2)
assert Q_t.tolist() == [[0, 1], [6, 7]], f'Q wrong: {Q_t.tolist()}'
assert K_t.tolist() == [[2, 3], [8, 9]], f'K wrong: {K_t.tolist()}'
assert V_t.tolist() == [[4, 5], [10, 11]], f'V wrong: {V_t.tolist()}'

# Apply to real qkv from c_attn — should match the inline slicing earlier
Q_r, K_r, V_r = split_qkv(qkv, d)
assert np.allclose(Q_r, Q) and np.allclose(K_r, K) and np.allclose(V_r, V), 'real qkv split mismatch'

# Round-trip: concatenating back should reproduce qkv
recombined = np.concatenate([Q_r, K_r, V_r], axis=-1)
assert np.allclose(recombined, qkv), 'concat(Q, K, V) should reproduce qkv'

print('split_qkv verified.')

In [ ]:
# Reshape (T, d) → (T, n_head, d_head) → (n_head, T, d_head)
def split_heads(x, n_head, d_head):
    T_ = x.shape[0]
    return x.reshape(T_, n_head, d_head).transpose(1, 0, 2)

Qh = split_heads(Q, n_head, d_head)        # (n_head, T, d_head)
Kh = split_heads(K, n_head, d_head)
Vh = split_heads(V, n_head, d_head)
print(f'per-head shapes: {Qh.shape}, {Kh.shape}, {Vh.shape}  ({n_head} heads × T × d_head)')

In [ ]:
# Per-head causal attention. Vectorised over heads with batched matmul.
M = causal_mask(T)                                      # (T, T) broadcasts across heads
scores = (Qh @ Kh.transpose(0, 2, 1)) / np.sqrt(d_head)  # (n_head, T, T)
scores = scores + M
A_per_head = softmax(scores, axis=-1)                   # (n_head, T, T)
head_out = A_per_head @ Vh                              # (n_head, T, d_head)

# Merge heads: (n_head, T, d_head) → (T, n_head, d_head) → (T, d)
merged = head_out.transpose(1, 0, 2).reshape(T, d)
print(f'after concat shape: {merged.shape}  (expect ({T}, {d}))')

# Output projection (c_proj is also Conv1D, same x @ W + b convention)
custom_attn_out = merged @ c_proj_w + c_proj_b           # (T, d)
print(f'custom attn output shape: {custom_attn_out.shape}')

## 7. Verify against HuggingFace

If the math (and the Conv1D convention) is right, our pure-NumPy output should match the HF model's attention sub-layer output to float32 precision. We allow `atol=1e-5` for accumulated rounding.

In [ ]:
hf = hf_attn_out[0]                                       # (T, d)
max_abs_diff = float(np.max(np.abs(custom_attn_out - hf)))
rel_err = max_abs_diff / float(np.max(np.abs(hf)))

print(f'max |custom - hf| = {max_abs_diff:.3e}')
print(f'relative max diff = {rel_err:.3e}')
print(f'np.allclose(atol=1e-5): {np.allclose(custom_attn_out, hf, atol=1e-5)}')
print(f'np.allclose(atol=1e-4): {np.allclose(custom_attn_out, hf, atol=1e-4)}')

assert np.allclose(custom_attn_out, hf, atol=1e-4), 'attention block mismatch'
print('\n  PASS — your NumPy attention reproduces the HuggingFace model to float-precision.')

### Your turn: package the attention block as one function

Wrap the entire NumPy attention sub-layer (LN → Conv1D → split → heads → causal attention → merge → output Conv1D) into a single function. This is the primitive you'd reach for if you wanted to apply your code to a different layer or a different model.

You can reuse `layernorm`, `causal_mask`, and `softmax` (defined in section 6) as well as `apply_conv1d` and `split_qkv` from your earlier exercises.

In [ ]:
def numpy_attention_block(
    hidden,
    ln_w, ln_b,
    c_attn_w, c_attn_b,
    c_proj_w, c_proj_b,
    n_head,
    eps=1e-5,
):
    """Run the GPT-2 attention sub-layer in pure NumPy.

    Args:
        hidden: (T, d) — residual-stream input to the layer (e.g. hidden_states[L] from HF)
        ln_w, ln_b: (d,) each — pre-attention LayerNorm parameters
        c_attn_w, c_attn_b: Q/K/V projection weight (d, 3d) and bias (3d,)
        c_proj_w, c_proj_b: output projection weight (d, d) and bias (d,)
        n_head: number of attention heads (d must be divisible by n_head)
        eps: LayerNorm epsilon (1e-5 in GPT-2 / distilgpt2)

    Returns:
        (T, d) output — what HF returns from h[layer].attn before the residual is added.

    The pipeline:
      1. h_ln = layernorm(hidden, ln_w, ln_b, eps)
      2. qkv = apply_conv1d(h_ln, c_attn_w, c_attn_b)        -- (T, 3d)
      3. Q, K, V = split_qkv(qkv, d)                          -- each (T, d)
      4. Reshape to (n_head, T, d_head) per head
      5. Per-head causal attention: softmax(QK^T / sqrt(d_head) + mask) @ V
      6. Merge back to (T, d)
      7. output = apply_conv1d(merged, c_proj_w, c_proj_b)
    """
    # TODO: assemble the block from the helpers above
    pass

In [ ]:
# --- TEST: numpy_attention_block ---

custom = numpy_attention_block(
    h, ln1_w, ln1_b, c_attn_w, c_attn_b, c_proj_w, c_proj_b,
    n_head=n_head, eps=ln_eps,
)

max_diff = float(np.max(np.abs(custom - hf)))
print(f'max |custom - hf| via numpy_attention_block = {max_diff:.3e}')

assert custom.shape == hf.shape, f'shape mismatch: {custom.shape} vs {hf.shape}'
assert np.allclose(custom, hf, atol=1e-4), f'numpy_attention_block does not match HF: {max_diff}'

print('numpy_attention_block reproduces the HuggingFace attention sub-layer.')

## 8. Visualise all 12 attention heads

Now the fun part — what does each head actually look at? We plot the per-head attention matrix `A_per_head` for layer 3 on the cat-sat-on-the-mat sentence. Heads often specialise: previous-token heads (sub-diagonal stripe), copy heads, BOS-attractor heads, syntactic heads.

We also confirm our `A_per_head` matches HuggingFace's reported attention weights for this layer.

In [ ]:
# Cross-check: do our per-head attention weights match HF's?
hf_A = hf_attn_weights[0]                  # (n_head, T, T)
max_A_diff = float(np.max(np.abs(A_per_head - hf_A)))
print(f'per-head attention weights vs HF: max diff = {max_A_diff:.3e}')
assert np.allclose(A_per_head, hf_A, atol=1e-5), 'per-head attention weight mismatch'

# Plot 12 heads in a 3x4 grid
fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for h_ix in range(n_head):
    ax = axes[h_ix // 4, h_ix % 4]
    im = ax.imshow(A_per_head[h_ix], cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'head {h_ix}')
    ax.set_xticks(range(T))
    ax.set_xticklabels([t.strip() for t in tokens], rotation=45, fontsize=8)
    ax.set_yticks(range(T))
    ax.set_yticklabels([t.strip() for t in tokens], fontsize=8)
fig.suptitle(f'distilgpt2 layer {LAYER} — attention patterns per head on "{SENTENCE}"',
             fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

## 9. Bonus — sampling next tokens

We have not built the FFN or the residual stream by hand — but HF has, and the final hidden state is sitting in `out.hidden_states[-1]`. Apply `model.lm_head` (which is weight-tied to the input embedding `wte`), sample with temperature + top-k, and read off the model's predictions for the next token.

This closes the lecture's full forward-pass story: tokens → embed → blocks → head → next-token distribution → sample.

In [ ]:
# Final hidden state (output of last layer) — for the LAST token only, since we want next-token logits
h_final = out.hidden_states[-1][0, -1].detach().cpu().numpy()    # (d,)
wte_w = model.transformer.wte.weight.detach().cpu().numpy()      # (V, d)

# Logits = h_final @ wte_w.T (weight-tied output head)
logits = h_final @ wte_w.T                                       # (V,)
print(f'logits shape: {logits.shape}  (V = vocab size)')

def sample_top_k(logits, temperature=0.8, top_k=10, n_samples=5, seed=0):
    rng_local = np.random.default_rng(seed)
    scaled = logits / temperature
    top_idx = np.argpartition(scaled, -top_k)[-top_k:]
    top_logits = scaled[top_idx]
    probs = np.exp(top_logits - top_logits.max())
    probs = probs / probs.sum()
    samples = rng_local.choice(top_idx, size=n_samples, p=probs)
    return samples, dict(zip(top_idx, probs))

samples, top_probs = sample_top_k(logits, temperature=0.8, top_k=10, n_samples=8)
print(f'\nTop-10 candidates after "{SENTENCE}" (temperature=0.8):')
for tid in sorted(top_probs, key=top_probs.get, reverse=True):
    print(f'  p={top_probs[tid]:.3f}  id={tid:>5d}  token={tokenizer.decode([int(tid)])!r}')

print(f'\n8 samples drawn:')
for tid in samples:
    print(f'  {SENTENCE} +  {tokenizer.decode([int(tid)])!r}')

## 10. Self-check + AI Diary

Answer each prompt below in the markdown cell underneath.

### 1. The Conv1D gotcha

Section 5 showed `c_attn.weight.shape == (d, 3d)` and the forward call is `x @ W + b`. If HuggingFace had used `nn.Linear` instead, what would the weight shape and forward call look like? In one sentence, why is this difference worth knowing about even outside this notebook?

*Your answer here*

### 2. Pick a head and describe it

Look at the 12 heatmaps in section 8. Pick **one** head whose pattern caught your eye and describe it in 2–3 sentences: which positions does each query attend to, and what does that pattern remind you of (previous-token, BOS-attractor, copy, syntactic, ...)?

*Your answer here*

### 3. Why we did not reproduce the full layer

We verified the **attention sub-layer** of layer 3 against HuggingFace, not the entire layer. To reproduce the *full* output of layer 3 to float-precision, what additional pieces would you need to implement (in order)? List them — you don't have to write the code.

*Your answer here*

### 4. Weight tying

Section 9 used `wte.weight.T` as the output projection. The lecture calls this *weight tying* and notes it saves `V · d` parameters compared to a separate `W_U`. For distilgpt2 (V ≈ 50 257, d = 768), how many parameters does weight tying save? Express the saving as a percentage of the model's total parameter count.

*Your answer here*

### 5. Temperature sweep

Re-run the sampling cell in section 9 with `temperature ∈ {0.3, 1.0, 2.0}`. Paste the top-10 distributions for each, and in 2–3 sentences explain what changed and why. Connect this to the lecture's discussion of greedy vs creative sampling.

*Your answer here*